# 01 source inspection

Record verified source formats, CRS, geometries, attributes, IDs and connectivity evidence.

In [1]:
from pathlib import Path
import zipfile

import fiona
import geopandas as gpd

In [2]:
ROOT = next(
      p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "data/raw").is_dir()
  )

In [3]:
vancouver = {
    path.stem: gpd.read_file(path) for path in sorted((ROOT / "data/raw/vancouver").glob("*.geojson"))
}

dnv = {}

for archive_path in sorted((ROOT / "data/raw/dnv").glob("*_fgdb.zip")):
    destination = ROOT / "data/interim/dnv" / archive_path.stem
    '''
    if not destination.exists():
        # Validate archive paths before extracting.
        with zipfile.ZipFile(archive_path) as archive:
            for name in archive.namelist():
                target = (destination / name).resolve()
                if not target.is_relative_to(destination.resolve()):
                    raise ValueError(f"Unsafe archive path: {name}")

            destination.mkdir(parents=True)
            archive.extractall(destination)
    '''
    for database in sorted(destination.rglob("*.gdb")):
        for layer in fiona.listlayers(database):
            key = f"{archive_path.stem}/{layer}"
            dnv[key] = gpd.read_file(database, layer=layer)

In [7]:
print(list(vancouver))
print(list(dnv))

['street-lighting-abandoned-conduits', 'street-lighting-conduits', 'street-lighting-junction-boxes', 'street-lighting-poles', 'street-lighting-service-panels']
['LgtStreetLightConduit_fgdb/LgtStreetLightConduit', 'LgtStreetLightFittings_fgdb/LgtStreetLightFittings', 'LgtStreetLightPoles_fgdb/LgtStreetLightPoles']


In [5]:
vancouver_poles = vancouver['street-lighting-poles']
vancouver_abandoned_conduits = vancouver['street-lighting-abandoned-conduits']
vancouver_conduits = vancouver['street-lighting-conduits']
vancouver_junction_boxes = vancouver['street-lighting-junction-boxes']
vancouver_service_panels = vancouver['street-lighting-service-panels']

In [6]:
dnv_conduit = dnv['LgtStreetLightConduit_fgdb/LgtStreetLightConduit']
dnv_poles = dnv['LgtStreetLightPoles_fgdb/LgtStreetLightPoles']
dnv_fittings = dnv['LgtStreetLightFittings_fgdb/LgtStreetLightFittings']

In [9]:
print(dnv_fittings.columns)

Index(['Asset_Id', 'Network_Id', 'AM_Owner', 'AM_Owner_resolved', 'AM_Dept',
       'AM_Dept_resolved', 'AM_Date', 'AM_Material', 'AM_Size', 'AM_Type',
       'AM_Estimated', 'Met_Gather', 'Met_Donor', 'Met_Donor_resolved',
       'Met_Format', 'Met_Format_resolved', 'Met_Input', 'Met_Tech',
       'Met_Tech_resolved', 'Met_Method', 'Met_Method_resolved', 'Met_Refile',
       'Comments', 'GlobalID', 'Contributed_Asset',
       'Contributed_Asset_resolved', 'Developer', 'Designer', 'CreatedUser',
       'CreatedDate', 'LastEditor', 'LastEditDate', 'Asset_Owner',
       'Asset_Owner_resolved', 'Asset_Manager', 'Asset_Manager_resolved',
       'Asset_Operator', 'Asset_Operator_resolved', 'Interim',
       'Interim_resolved', 'Lifecycle_Status', 'Lifecycle_Status_resolved',
       'Condition_Date', 'Condition_Score', 'geometry'],
      dtype='object')


In [14]:
for name, layer in dnv.items():
    print(name, "=>", layer.crs)

for name, layer in vancouver.items():
    print(name, "=>", layer.crs)

LgtStreetLightConduit_fgdb/LgtStreetLightConduit => EPSG:26910
LgtStreetLightFittings_fgdb/LgtStreetLightFittings => EPSG:26910
LgtStreetLightPoles_fgdb/LgtStreetLightPoles => EPSG:26910
street-lighting-abandoned-conduits => EPSG:4326
street-lighting-conduits => EPSG:4326
street-lighting-junction-boxes => EPSG:4326
street-lighting-poles => EPSG:4326
street-lighting-service-panels => EPSG:4326


In [15]:
for name, layer in dnv.items():
    print(name, layer.geom_type.value_counts().to_dict())

for name, layer in vancouver.items():
    print(name, layer.geom_type.value_counts().to_dict())

LgtStreetLightConduit_fgdb/LgtStreetLightConduit {'MultiLineString': 4641}
LgtStreetLightFittings_fgdb/LgtStreetLightFittings {'Point': 1576}
LgtStreetLightPoles_fgdb/LgtStreetLightPoles {'Point': 5474}
street-lighting-abandoned-conduits {'LineString': 774}
street-lighting-conduits {'LineString': 65429, 'Point': 16, 'MultiLineString': 11}
street-lighting-junction-boxes {'Point': 7947}
street-lighting-poles {'Point': 57984}
street-lighting-service-panels {'Point': 1455}


In [17]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 200)
pd.set_option("display.expand_frame_repr", False)

In [22]:
for name, layer in vancouver.items():
  print(f"\n{name}")
  print("Columns:", layer.columns.tolist())
  display(layer.head(3))
  layer.info()


street-lighting-abandoned-conduits
Columns: ['wireset_type', 'geo_point_2d', 'geometry']


,wireset_type,geo_point_2d,geometry
0,3-4,"{'lon': -123.17324858943866, 'lat': 49.264217636860636}","LINESTRING (-123.17346 49.26422, -123.17304 49.26421)"
1,3-6,"{'lon': -123.16389793278195, 'lat': 49.26393875976352}","LINESTRING (-123.16408 49.26394, -123.16372 49.26393)"
2,3-8,"{'lon': -123.11777959632627, 'lat': 49.2452332215235}","LINESTRING (-123.11794 49.24518, -123.11762 49.24529)"


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 774 entries, 0 to 773
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   wireset_type  680 non-null    object  
 1   geo_point_2d  774 non-null    object  
 2   geometry      774 non-null    geometry
dtypes: geometry(1), object(2)
memory usage: 18.3+ KB

street-lighting-conduits
Columns: ['wireset_type', 'geo_point_2d', 'geometry']


,wireset_type,geo_point_2d,geometry
0,3-10,"{'lon': -123.1690665273631, 'lat': 49.2261957337646}","LINESTRING (-123.16926 49.22630, -123.16887 49.22609)"
1,2-10,"{'lon': -123.15901378502336, 'lat': 49.222133564088566}","LINESTRING (-123.15904 49.22215, -123.15899 49.22212)"
2,3-10,"{'lon': -123.15910318243768, 'lat': 49.22266666280071}","LINESTRING (-123.15910 49.22251, -123.15911 49.22283)"


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 65456 entries, 0 to 65455
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   wireset_type  58404 non-null  object  
 1   geo_point_2d  65456 non-null  object  
 2   geometry      65456 non-null  geometry
dtypes: geometry(1), object(2)
memory usage: 1.5+ MB

street-lighting-junction-boxes
Columns: ['material', 'geo_local_area', 'geo_point_2d', 'geometry']


...

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 7947 entries, 0 to 7946
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   material        1568 non-null   object  
 1   geo_local_area  7903 non-null   object  
 2   geo_point_2d    7947 non-null   object  
 3   geometry        7947 non-null   geometry
dtypes: geometry(1), object(3)
memory usage: 248.5+ KB

street-lighting-poles
Columns: ['block_number', 'node_number', 'geo_local_area', 'geo_point_2d', 'geometry']


,block_number,node_number,geo_local_area,geo_point_2d,geometry
0,70,5.0,Victoria-Fraserview,"{'lon': -123.06247569245082, 'lat': 49.23075945327847}",POINT (-123.06248 49.23076)
1,29,1.0,Kitsilano,"{'lon': -123.17105419341743, 'lat': 49.259651226369954}",POINT (-123.17105 49.25965)
2,36,2.0,Dunbar-Southlands,"{'lon': -123.17846017286428, 'lat': 49.25417478927384}",POINT (-123.17846 49.25417)


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 57984 entries, 0 to 57983
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   block_number    57975 non-null  object  
 1   node_number     57984 non-null  float64 
 2   geo_local_area  57681 non-null  object  
 3   geo_point_2d    57984 non-null  object  
 4   geometry        57984 non-null  geometry
dtypes: float64(1), geometry(1), object(3)
memory usage: 2.2+ MB

street-lighting-service-panels
Columns: ['service_panel_number', 'geo_local_area', 'geo_point_2d', 'geometry']


,service_panel_number,geo_local_area,geo_point_2d,geometry
0,6957PE,Victoria-Fraserview,"{'lon': -123.06620870661031, 'lat': 49.23249079562975}",POINT (-123.06621 49.23249)
1,U0538CEBLR,South Cambie,"{'lon': -123.11608668624932, 'lat': 49.251330373091385}",POINT (-123.11609 49.25133)
2,8449DE,Renfrew-Collingwood,"{'lon': -123.0326498673338, 'lat': 49.240510609155564}",POINT (-123.03265 49.24051)


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1455 entries, 0 to 1454
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   service_panel_number  1453 non-null   object  
 1   geo_local_area        1450 non-null   object  
 2   geo_point_2d          1455 non-null   object  
 3   geometry              1455 non-null   geometry
dtypes: geometry(1), object(3)
memory usage: 45.6+ KB


In [18]:
for name, layer in dnv.items():
  print(f"\n{name}")
  print("Columns:", layer.columns.tolist())
  display(layer.head(3))
  layer.info()


LgtStreetLightConduit_fgdb/LgtStreetLightConduit
Columns: ['Asset_Id', 'Network_Id', 'AM_Owner', 'AM_Owner_resolved', 'AM_Dept', 'AM_Dept_resolved', 'AM_Date', 'AM_Material', 'AM_Size', 'AM_Type', 'AM_Estimated', 'Met_Gather', 'Met_Donor', 'Met_Donor_resolved', 'Met_Format', 'Met_Format_resolved', 'Met_Input', 'Met_Tech', 'Met_Tech_resolved', 'Met_Method', 'Met_Method_resolved', 'Met_Refile', 'ConduitStatus', 'ConduitStatus_resolved', 'Comments', 'GlobalID', 'Contributed_Asset', 'Contributed_Asset_resolved', 'Developer', 'Designer', 'CreatedUser', 'CreatedDate', 'LastEditor', 'LastEditDate', 'Asset_Owner', 'Asset_Owner_resolved', 'Asset_Manager', 'Asset_Manager_resolved', 'Asset_Operator', 'Asset_Operator_resolved', 'Interim', 'Interim_resolved', 'Lifecycle_Status', 'Lifecycle_Status_resolved', 'Condition_Date', 'Condition_Score', 'SHAPE_Length', 'geometry']


,Asset_Id,Network_Id,AM_Owner,AM_Owner_resolved,AM_Dept,AM_Dept_resolved,AM_Date,AM_Material,AM_Size,AM_Type,AM_Estimated,Met_Gather,Met_Donor,Met_Donor_resolved,Met_Format,Met_Format_resolved,Met_Input,Met_Tech,Met_Tech_resolved,Met_Method,Met_Method_resolved,Met_Refile,ConduitStatus,ConduitStatus_resolved,Comments,GlobalID,Contributed_Asset,Contributed_Asset_resolved,Developer,Designer,CreatedUser,CreatedDate,LastEditor,LastEditDate,Asset_Owner,Asset_Owner_resolved,Asset_Manager,Asset_Manager_resolved,Asset_Operator,Asset_Operator_resolved,Interim,Interim_resolved,Lifecycle_Status,Lifecycle_Status_resolved,Condition_Date,Condition_Score,SHAPE_Length,geometry
0,LGTCON00010,LGTNET00185,DNV,DISTRICT OF NORTH VANCOUVER,TRAFFIC_DEPT,TRAFFIC_DEPT,1967.0,PVC,NaN,1,None,None,,,ENG DRAWING,ENG DRAWING,OCTOBER 2007,I JAMAL,I JAMAL,SOFTDIGITIZED,SOFTDIGITIZED,UF22,EXACT,EXACT,LGTNET00001,{6BC4D442-6207-47DD-9E11-39362B4BCE26},None,None,None,None,None,NaT,SDE_FERENCEK,2018-01-10 23:41:58+00:00,None,None,None,None,None,None,None,None,None,None,NaT,None,16.499892,"MULTILINESTRING ((497235.085 5464509.157, 497235.776 5464525.642))"
1,LGTCON00011,LGTNET00185,DNV,DISTRICT OF NORTH VANCOUVER,TRAFFIC_DEPT,TRAFFIC_DEPT,1967.0,PVC,NaN,1,None,None,,,ENG DRAWING,ENG DRAWING,OCTOBER 2007,I JAMAL,I JAMAL,SOFTDIGITIZED,SOFTDIGITIZED,UF22,EXACT,EXACT,LGTNET00001,{C87A34F8-7D7B-4A92-B7C3-1E1592E833DE},None,None,None,None,None,NaT,SDE_FERENCEK,2018-01-10 23:41:58+00:00,None,None,None,None,None,None,None,None,None,None,NaT,None,69.470826,"MULTILINESTRING ((497235.085 5464509.157, 497237.089 5464508.981, 497238.548 5464508.510, 497239..."
2,LGTCON00017,LGTNET00258,DNV,DISTRICT OF NORTH VANCOUVER,TRAFFIC_DEPT,TRAFFIC_DEPT,1967.0,PVC,NaN,1,None,None,,,ENG DRAWING,ENG DRAWING,OCTOBER 2007,I JAMAL,I JAMAL,SOFTDIGITIZED,SOFTDIGITIZED,UF22,EXACT,EXACT,LGTNET00348,{7FFD4767-FD06-4A09-9010-CBFACE712ED0},None,None,None,None,None,NaT,SDE_FERENCEK,2018-01-10 23:41:58+00:00,None,None,None,None,None,None,None,None,None,None,NaT,None,47.791071,"MULTILINESTRING ((496805.497 5464513.090, 496806.198 5464512.858, 496842.298 5464543.037))"


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 4641 entries, 0 to 4640
Data columns (total 48 columns):
 #   Column                      Non-Null Count  Dtype              
---  ------                      --------------  -----              
 0   Asset_Id                    4641 non-null   object             
 1   Network_Id                  4641 non-null   object             
 2   AM_Owner                    4641 non-null   object             
 3   AM_Owner_resolved           4641 non-null   object             
 4   AM_Dept                     4607 non-null   object             
 5   AM_Dept_resolved            4607 non-null   object             
 6   AM_Date                     4637 non-null   float64            
 7   AM_Material                 4611 non-null   object             
 8   AM_Size                     904 non-null    float64            
 9   AM_Type                     4641 non-null   int64              
 10  AM_Estimated                170 non-null    object  

,Asset_Id,Network_Id,AM_Owner,AM_Owner_resolved,AM_Dept,AM_Dept_resolved,AM_Date,AM_Material,AM_Size,AM_Type,AM_Estimated,Met_Gather,Met_Donor,Met_Donor_resolved,Met_Format,Met_Format_resolved,Met_Input,Met_Tech,Met_Tech_resolved,Met_Method,Met_Method_resolved,Met_Refile,Comments,GlobalID,Contributed_Asset,Contributed_Asset_resolved,Developer,Designer,CreatedUser,CreatedDate,LastEditor,LastEditDate,Asset_Owner,Asset_Owner_resolved,Asset_Manager,Asset_Manager_resolved,Asset_Operator,Asset_Operator_resolved,Interim,Interim_resolved,Lifecycle_Status,Lifecycle_Status_resolved,Condition_Date,Condition_Score,geometry
0,LGTFIT00001,LGTNET00115,DNV,DISTRICT OF NORTH VANCOUVER,DESIGN_DEPT,DESIGN_DEPT,1979.0,None,None,2,None,None,,,ENG DRAWING,ENG DRAWING,DECEMBER 2007,I JAMAL,I JAMAL,SOFTDIGITIZED,SOFTDIGITIZED,None,Service Box,{B192FFA2-CB4C-4244-88A8-2BE4CF229B05},None,None,None,None,None,NaT,None,NaT,None,None,None,None,None,None,None,None,None,None,NaT,None,POINT (492638.671 5467928.097)
1,LGTFIT00002,LGTNET00106,BC HYDRO,BC HYDRO,None,None,1979.0,None,None,2,None,None,,,ENG DRAWING,ENG DRAWING,DECEMBER 2007,I JAMAL,I JAMAL,SOFTDIGITIZED,SOFTDIGITIZED,None,BC Hydro Service Box,{B4F444F4-39F5-4AE0-825D-747948886606},None,None,None,None,None,NaT,SDE_MARUTA,2022-04-04 16:44:50+00:00,None,None,None,None,None,None,None,None,None,None,NaT,None,POINT (492453.537 5467762.901)
2,LGTFIT00003,LGTNET00104,BC HYDRO,BC HYDRO,None,None,1979.0,None,None,2,None,None,,,ENG DRAWING,ENG DRAWING,DECEMBER 2007,I JAMAL,I JAMAL,SOFTDIGITIZED,SOFTDIGITIZED,None,BC Hydro Service Box,{47A5082A-29E7-45B6-A7A2-0C54629DCB36},None,None,None,None,None,NaT,SDE_MARUTA,2022-04-04 16:44:54+00:00,None,None,None,None,None,None,None,None,None,None,NaT,None,POINT (492449.889 5467660.874)


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1576 entries, 0 to 1575
Data columns (total 45 columns):
 #   Column                      Non-Null Count  Dtype              
---  ------                      --------------  -----              
 0   Asset_Id                    1576 non-null   object             
 1   Network_Id                  1551 non-null   object             
 2   AM_Owner                    1571 non-null   object             
 3   AM_Owner_resolved           1571 non-null   object             
 4   AM_Dept                     1301 non-null   object             
 5   AM_Dept_resolved            1301 non-null   object             
 6   AM_Date                     1548 non-null   float64            
 7   AM_Material                 0 non-null      object             
 8   AM_Size                     0 non-null      object             
 9   AM_Type                     1576 non-null   int64              
 10  AM_Estimated                13 non-null     object  

,Asset_Id,Network_Id,AM_Owner,AM_Owner_resolved,AM_Dept,AM_Dept_resolved,AM_Type,AM_Estimated,Standard_Date,Foundation_Type,Pole_Installed,Pole_ID,Pole_Type,Pole_Type_resolved,Pole_Subtype,Pole_Colour,Pole_Material,Pole_Height,Pole_Shape,Pole_Arm,Pole_ArmLength,Pole_ArmLength_resolved,Pole_DistributionBase,Pole_DistributionBase_resolved,Pole_Last_DBMaintenance,Pole_Last_Powdercoat,Pole_Last_Painting,Luminaire_Installed,Luminaire_Type,Lamp_Installed,Lamp_Type,Lamp_Wattage,Circuit_Number,Circuit_Quadrant,Circuit_Quadrant_resolved,Circuit_ServiceLocation,Hydro_Code,Hydro_Code_resolved,Hydro_FixtureNumber,Road_Orientation_Angle,Road_Orientation_Angle_resolved,Met_Gather,Met_Donor,Met_Donor_resolved,Met_Format,Met_Format_resolved,Met_Input,Met_Tech,Met_Tech_resolved,Met_Method,Met_Method_resolved,Met_Refile,Comments,Pole_Receptacle,Pole_Receptacle_resolved,Under_Warranty,Under_Warranty_resolved,Contributed_Asset,Contributed_Asset_resolved,Developer,Designer,CreatedUser,CreatedDate,LastEditor,LastEditDate,Luminaire_manufacturer,Luminaire_model,Lamp_Temperature,Luminaire_Shield,Luminaire_Shield_resolved,Lamp_Base,Ladder_Access,Ladder_Access_resolved,Fixture_Type,Fixture_Type_resolved,Driver_Setting,Driver_Setting_resolved,Fixture_InstallDate,Ground_Wire,Ground_Wire_resolved,FixtureComments,MRN,MRN_resolved,RoadClass,RoadClass_resolved,PhotoCell,PhotoCell_resolved,LEDShield,LEDShield_resolved,Banner,Banner_resolved,Size,GlobalID,Asset_Owner,Asset_Owner_resolved,Asset_Manager,Asset_Manager_resolved,Asset_Operator,Asset_Operator_resolved,Interim,Interim_resolved,Lifecycle_Status,Lifecycle_Status_resolved,Condition_Date,Condition_Score,ServiceBaseInsp,geometry
0,LGTLT00001,LGTNET00228,DNV,DISTRICT OF NORTH VANCOUVER,TRAFFIC_DEPT,TRAFFIC_DEPT,1,None,2003.0,,2003.0,2655,1,STANDARD,COBRA 2 SHAFT,GREEN,METAL,9.0,OCTAGONAL,None,None,None,N,NO,NaN,2005.0,2026.0,2019.0,COBRA FLAT LENS,2019.0,LED,28.0,None,2,2,None,None,None,None,None,None,None,N/A,N/A,ENG DRAWING,ENG DRAWING,DECEMBER 2007,I JAMAL,I JAMAL,SOFTDIGITIZED,SOFTDIGITIZED,UF496,None,None,None,None,None,None,None,None,None,None,NaT,tomvandas,2026-08-08 23:19:55+00:00,LED ROADWAY LIGHTING,NXT-24S-0-7-2ES-3-GY-4-UL-S-2H,3000.0,N,NO,None,None,None,A,A,3,3,2019-01-17 20:29:20+00:00,Y,YES,None,None,None,5,LOCAL,N,NO,N,NO,None,None,None,{91054AD8-869F-43F2-9887-5ED84D6C85B2},Transportation,Transportation,Traffic Ops,Traffic Ops,Traffic Ops,Traffic Ops,N,NO,Active,Active,NaT,None,NaN,POINT (497261.759 5466622.663)
1,LGTLT00002,LGTNET00228,DNV,DISTRICT OF NORTH VANCOUVER,TRAFFIC_DEPT,TRAFFIC_DEPT,1,None,2003.0,,2003.0,2656,1,STANDARD,COBRA 2 SHAFT,GREEN,METAL,7.5,OCTAGONAL,None,None,None,N,NO,NaN,2005.0,2025.0,2019.0,COBRA FLAT LENS,2019.0,LED,28.0,None,2,2,None,None,None,None,None,None,None,N/A,N/A,ENG DRAWING,ENG DRAWING,DECEMBER 2007,I JAMAL,I JAMAL,SOFTDIGITIZED,SOFTDIGITIZED,UF496,None,None,None,None,None,None,None,None,None,None,NaT,SDE_MARUTA,2026-03-26 16:49:36+00:00,LED ROADWAY LIGHTING,NXT-24S-0-7-2ES-3-GY-4-UL-S-2H,3000.0,N,NO,None,None,None,A,A,3,3,2019-01-17 20:29:02+00:00,Y,YES,None,None,None,5,LOCAL,Y,YES,N,NO,None,None,None,{74B76536-F324-4DEE-AC08-D22FC7E5D274},Transportation,Transportation,Traffic Ops,Traffic Ops,Traffic Ops,Traffic Ops,N,NO,Active,Active,NaT,None,NaN,POINT (497248.516 5466634.782)
2,LGTLT00003,LGTNET00228,DNV,DISTRICT OF NORTH VANCOUVER,TRAFFIC_DEPT,TRAFFIC_DEPT,1,None,2003.0,,2003.0,2657,1,STANDARD,COBRA 2 SHAFT,GREEN,METAL,7.5,OCTAGONAL,None,None,None,Y,YES,NaN,2005.0,2026.0,2019.0,COBRA FLAT LENS,2019.0,LED,28.0,None,2,2,None,None,None,None,None,None,None,N/A,N/A,ENG DRAWING,ENG DRAWING,DECEMBER 2007,I JAMAL,I JAMAL,SOFTDIGITIZED,SOFTDIGITIZED,UF496,None,None,None,None,None,None,None,None,None,None,NaT,tomvandas,2026-08-08 23:19:45+00:00,LED ROADWAY LIGHTING,NXT-24S-0-7-2ES-3-GY-4-UL-S-2H,3000.0,N,NO,None,None,None,A,A,3,3,2019-01-17 20:28:34+00:00,Y,YES,None,None,None,5,LOCAL,N,NO,N,NO,None,None,None,{F387E706-2BDF-4BD6-A3FC-1BD2B3D25C66},Transportation,

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 5474 entries, 0 to 5473
Columns: 107 entries, Asset_Id to geometry
dtypes: datetime64[ns, UTC](3), datetime64[ns](1), float64(11), geometry(1), int64(1), object(90)
memory usage: 4.5+ MB


In [21]:
for name, layer in vancouver.items():
    print(f"\n{name}")
    print([
      column for column in layer.columns
      if any(word in column.lower()
             for word in ["id", "node", "network", "asset"])
    ])


street-lighting-abandoned-conduits
[]

street-lighting-conduits
[]

street-lighting-junction-boxes
[]

street-lighting-poles
['node_number']

street-lighting-service-panels
[]


In [ ]:
for left_name, left in vancouver.items():
    for right_name, right in vancouver.items():
      if left_name >= right_name:
          continue
    
      common = set(left.columns) & set(right.columns)
    
      for column in common:
          if "id" in column.lower() or "network" in column.lower():
              overlap = (
                  set(left[column].dropna().astype(str))
                  & set(right[column].dropna().astype(str))
              )
              print(left_name, "<->", right_name, column,
                    "overlap:", len(overlap))

In [19]:
for name, layer in dnv.items():
    print(f"\n{name}")
    print([
      column for column in layer.columns
      if any(word in column.lower()
             for word in ["id", "node", "network", "asset"])
    ])


LgtStreetLightConduit_fgdb/LgtStreetLightConduit
['Asset_Id', 'Network_Id', 'GlobalID', 'Contributed_Asset', 'Contributed_Asset_resolved', 'Asset_Owner', 'Asset_Owner_resolved', 'Asset_Manager', 'Asset_Manager_resolved', 'Asset_Operator', 'Asset_Operator_resolved']

LgtStreetLightFittings_fgdb/LgtStreetLightFittings
['Asset_Id', 'Network_Id', 'GlobalID', 'Contributed_Asset', 'Contributed_Asset_resolved', 'Asset_Owner', 'Asset_Owner_resolved', 'Asset_Manager', 'Asset_Manager_resolved', 'Asset_Operator', 'Asset_Operator_resolved']

LgtStreetLightPoles_fgdb/LgtStreetLightPoles
['Asset_Id', 'Network_Id', 'Pole_ID', 'Contributed_Asset', 'Contributed_Asset_resolved', 'GlobalID', 'Asset_Owner', 'Asset_Owner_resolved', 'Asset_Manager', 'Asset_Manager_resolved', 'Asset_Operator', 'Asset_Operator_resolved']


In [20]:
for left_name, left in dnv.items():
    for right_name, right in dnv.items():
      if left_name >= right_name:
          continue
    
      common = set(left.columns) & set(right.columns)
    
      for column in common:
          if "id" in column.lower() or "network" in column.lower():
              overlap = (
                  set(left[column].dropna().astype(str))
                  & set(right[column].dropna().astype(str))
              )
              print(left_name, "<->", right_name, column,
                    "overlap:", len(overlap))

LgtStreetLightConduit_fgdb/LgtStreetLightConduit <-> LgtStreetLightFittings_fgdb/LgtStreetLightFittings Asset_Id overlap: 0
LgtStreetLightConduit_fgdb/LgtStreetLightConduit <-> LgtStreetLightFittings_fgdb/LgtStreetLightFittings Network_Id overlap: 449
LgtStreetLightConduit_fgdb/LgtStreetLightConduit <-> LgtStreetLightFittings_fgdb/LgtStreetLightFittings GlobalID overlap: 0
LgtStreetLightConduit_fgdb/LgtStreetLightConduit <-> LgtStreetLightPoles_fgdb/LgtStreetLightPoles Asset_Id overlap: 0
LgtStreetLightConduit_fgdb/LgtStreetLightConduit <-> LgtStreetLightPoles_fgdb/LgtStreetLightPoles Network_Id overlap: 528
LgtStreetLightConduit_fgdb/LgtStreetLightConduit <-> LgtStreetLightPoles_fgdb/LgtStreetLightPoles GlobalID overlap: 0
LgtStreetLightFittings_fgdb/LgtStreetLightFittings <-> LgtStreetLightPoles_fgdb/LgtStreetLightPoles Asset_Id overlap: 0
LgtStreetLightFittings_fgdb/LgtStreetLightFittings <-> LgtStreetLightPoles_fgdb/LgtStreetLightPoles Network_Id overlap: 448
LgtStreetLightFittings